# 10 · Genomic Filtering

This notebook teaches the genomic-filtering API added to the Python adapter. By the end you'll be able to filter a query by genetic **variant annotations** (gene, variant consequence, population frequency) and pull back either **patient** counts or **variant**-level results.

### The mental model

A PIC-SURE query filters along two independent axes that combine with AND:

1. **Phenotypic** — clinical/observational traits (age, sex, a diagnosis). These are the `Clause` / `ClauseGroup` objects from `buildClause()` / `buildClauseGroup()`.
2. **Genomic** — properties of the genetic *variants* a participant carries. These are `GenomicFilter` objects from `buildGenomicFilter()`.

Unlike the phenotypic side (an AND/OR *tree*), genomic filters are a **flat list applied conjunctively** — every filter in the list must match. A query can use either axis, or both.

> Many cells below (building filters, inspecting `.to_query_json()`, the value enums) run **offline** — you don't need a live connection to learn the shapes. The cells that call `runQuery()` need a connection to a deployment that has genomic data.

## Setup

Genomic data lives on **authorized** BDC, so connect with a token. Put your PIC-SURE token in `notebooks/token.txt` (one line).

In [ ]:
import picsure

In [ ]:
token_file = "token.txt"

with open(token_file, "r") as f:
    my_token = f.read()

In [ ]:
session = picsure.connect(
    picsure.Platform.BDC_AUTHORIZED,
    my_token,
)

### Configure the examples

Genomic annotation **keys** and **values** are deployment-specific. The keys below (`Gene_with_variant`, `Variant_consequence_calculated`, `Variant_frequency_as_text`) are the standard BDC annotations; adjust the gene symbol to match your data. You can discover valid genes and consequences from the adapter (next section) or in the PIC-SURE UI's variant explorer.

In [ ]:
# --- edit these to match your deployment / cohort of interest ---
GENE = "CHD8"                 # a gene symbol present in the variant data
SEARCH_TERM = "sex"           # any phenotypic concept you can filter on

# Standard BDC genomic annotation keys (usually stable):
GENE_KEY = "Gene_with_variant"
CONSEQUENCE_KEY = "Variant_consequence_calculated"
FREQ_BUCKET_KEY = "Variant_frequency_as_text"

### Discovering valid genes & consequences

You don't have to hardcode genes and consequence values. On an **authorized** platform, `session.searchGenomicValues(key, query=...)` looks them up from the server — paginated and searchable like the UI's autocomplete (raise `size` to pull more per call; the total match count is on `df.attrs["total"]`). It works for any genomic key. For consequences there's also `picsure.genomicConsequences()`, the built-in vocabulary grouped by severity, which needs no connection.

In [ ]:
# Search genes with variants by prefix (authorized platforms only)
genes = session.searchGenomicValues(GENE_KEY, query="BRCA", size=20)
print(f"{genes.attrs['total']} genes match 'BRCA'")
genes

In [ ]:
# searchGenomicValues works for any genomic key — e.g. consequence values:
session.searchGenomicValues(CONSEQUENCE_KEY, query="splice")

In [ ]:
# The built-in consequence vocabulary, grouped by severity (offline, no server):
picsure.genomicConsequences()

## 1. Anatomy of a genomic filter

`buildGenomicFilter(key, values=...)` builds one filter. A filter is **categorical**: you name the annotation (`key`) and the `values` that count as a match. Build a gene filter and inspect the exact JSON it sends over the wire with `.to_query_json()` (no server needed):

In [ ]:
gene_filter = picsure.buildGenomicFilter(GENE_KEY, values=[GENE])
gene_filter.to_query_json()

`values` accepts a single string or a list — pass several values to match any of them. For example, a few variant-consequence categories at once:

In [ ]:
consequences = picsure.buildGenomicFilter(
    CONSEQUENCE_KEY, values=["missense_variant", "stop_gained"]
)
consequences.to_query_json()

### Value enum

A small fixed vocabulary ships as an enum for discoverability: `VariantFrequency` (Rare/Common/Novel buckets for the `Variant_frequency_as_text` key). Pass a member **directly** to `values=` — the builder uses its `.value`.

> Heads up: this is a `str`-Enum, so `str(VariantFrequency.RARE)` is `'VariantFrequency.RARE'`, not `'Rare'`. If you need the raw wire string yourself, use `.value`. Passing the member to `buildGenomicFilter` always does the right thing.

In [ ]:
from picsure import VariantFrequency

print([m.value for m in VariantFrequency])

In [ ]:
rare_bucket = picsure.buildGenomicFilter(
    FREQ_BUCKET_KEY, values=VariantFrequency.RARE
)
rare_bucket.to_query_json()

## 2. A genomic filter as a constraint on a patient count

Attach a genomic filter to a query and run it like any other. With the patient-centric result types (`count`, `participant`), the genomic filter acts as a **constraint**: *participants who carry at least one matching variant*. A query may be **genomic-only** — no phenotypic filter required.

In [ ]:
gene_query = picsure.buildQuery(genomicFilters=gene_filter)

count = session.runQuery(gene_query, type="count")
count.value

`count` is a `CountResult`. On small cohorts the server obfuscates the number, so check the fields rather than assuming `value` is set:

In [ ]:
if count.value is not None:
    print(f"{count.value} participants carry a qualifying {GENE} variant")
else:
    print(f"fewer than {count.cap} participants (suppressed)")

## 3. Combining genomic + phenotypic filters

Pass both `phenotypicFilter=` and `genomicFilters=` to `buildQuery()`. They combine with AND: *matches the phenotype **and** carries a matching variant.* Here we find any phenotypic concept by search and use a `REQUIRE` clause (matches participants who have a value for it):

In [ ]:
results = session.searchDictionary(SEARCH_TERM)
concept_path = results["conceptPath"].iloc[0]
concept_path

In [ ]:
phenotypic = picsure.buildClause(
    concept_path, type=picsure.PhenotypicFilterType.REQUIRE
)

combined = picsure.buildQuery(
    phenotypicFilter=phenotypic,
    genomicFilters=gene_filter,
)

session.runQuery(combined, type="count").value

## 4. Multiple genomic filters (ANDed)

Pass a **list** to narrow the variant set further — e.g. *rare, loss-of-function variants in our gene*. Each entry must match:

In [ ]:
filters = [
    picsure.buildGenomicFilter(GENE_KEY, values=[GENE]),
    picsure.buildGenomicFilter(
        CONSEQUENCE_KEY, values=["stop_gained", "frameshift_variant"]
    ),
    picsure.buildGenomicFilter(FREQ_BUCKET_KEY, values=VariantFrequency.RARE),
]

multi = picsure.buildQuery(genomicFilters=filters)
[f.to_query_json() for f in multi.genomicFilters]

In [ ]:
session.runQuery(multi, type="count").value

## 5. Variant-centric result types

The result types above answer questions about **people**. Four new result types answer questions about the **variants** themselves:

| `type=`                   | returns          | question |
|---------------------------|------------------|----------|
| `"variant_count"`         | `CountResult`    | how many distinct variants match? |
| `"variant_list"`          | `list[str]`      | which variants match? |
| `"vcf_excerpt"`           | `DataFrame`      | a VCF-style table, **with** per-patient genotype columns |
| `"aggregate_vcf_excerpt"` | `DataFrame`      | same table, **without** patient columns (privacy-preserving) |

All four respect the same genomic + phenotypic filters.

> **Deployment note (as of 2026-06):** BDC's hosted PIC-SURE does not serve these variant result types yet — it returns an empty response, which the adapter surfaces as a clear `PicSureQueryError` ("...not available on this PIC-SURE deployment yet"). Genomic filtering as a **constraint** (Sections 2–4, using `count` / `participant`) works today. The cells below will raise that error on BDC until the backend enables variant output; they're shown so you know the API.

In [ ]:
# How many distinct variants match? (a CountResult, like a patient count)
variant_count = session.runQuery(multi, type="variant_count")
variant_count.value

`variant_list` returns the matching variant specs. Each spec is six comma-separated fields: `chromosome,offset,ref,alt,gene,consequence`.

In [ ]:
variants = session.runQuery(gene_query, type="variant_list")
print(f"{len(variants)} variants")
variants[:5]

In [ ]:
# VCF excerpt: one row per variant, with per-patient genotype columns.
vcf = session.runQuery(gene_query, type="vcf_excerpt")
vcf.head()

In [ ]:
# Aggregate VCF excerpt: same rows, without the per-patient columns.
agg = session.runQuery(gene_query, type="aggregate_vcf_excerpt")
agg.head()

## 6. Saving & loading genomic queries

Genomic filters survive a round-trip. On an authorized platform you can `saveQueryByName()` a query that carries genomic filters and later `loadQueryByID()` it back into an equivalent `Query` — including gene/consequence/frequency queries originally built in the PIC-SURE UI's variant explorer.

```python
qid = session.saveQueryByName(multi, "Rare LoF in my gene")
restored = session.loadQueryByID(qid)
restored.genomicFilters
```

## Recap / cheat sheet

- **Discover values:** `session.searchGenomicValues(key, query=...)` (authorized; genes, consequences, any genomic key) and `picsure.genomicConsequences()` (offline consequence vocabulary by severity).
- **Build one filter:** `picsure.buildGenomicFilter(key, values=...)` — one value or a list; matches any of them.
- **Attach to a query:** `buildQuery(genomicFilters=one_or_a_list)`; combine with `phenotypicFilter=` to AND the two axes. Genomic-only queries are fine.
- **Patient results** (`count`, `participant`): the genomic filter is a constraint on *who* matches.
- **Variant results** (`variant_count`, `variant_list`, `vcf_excerpt`, `aggregate_vcf_excerpt`): answers about the *variants*.
- **Enum:** `VariantFrequency` (Rare/Common/Novel) — pass members straight to `values=`.
- **Gotchas:** `variant_count` is a `CountResult` (handles obfuscation), not a bare int; `variant_list` specs are comma-delimited `chrom,offset,ref,alt,gene,consequence`; keys/genes are deployment-specific — discover them with `searchGenomicValues()` or the PIC-SURE UI. Variant-spec (SNP) filtering is not supported yet.